# Mapping Your Data to FHIR

Your data is not FHIR, but CQL reads FHIR resources. This notebook maps a small table of visits, in a made-up
EHR's own shape, to the FHIR resources a CQL library reads, runs the library over the mapping, and then checks the
mapping: an independent SQL query over the source must give the same answers as CQL.

1. **Source data:** patients and visits with local codes.
2. **Mapping:** one SQL view per FHIR type, each returning a `resource` struct in the FHIR R4 shape.
3. **CQL:** a library over `Patient` and `Encounter`, bound to the views.
4. **Check:** the same questions asked of the source in plain SQL; every patient must agree.

Data: four synthetic patients written in the notebook. No real patient data.

**Runs on serverless.** The client comes from this notebook's base environment, **CQL on Databricks**, so there is
no install cell.

In [0]:
%reload_ext cql_databricks.compiler
from cql_databricks.compiler import cql

## 1. The source data

Two tables in the EHR's own shape. Nothing here is FHIR: the visit types and statuses are local codes, and times
are timestamps.

| Table | Columns |
|---|---|
| `ehr_patients` | `mrn`, `sex` (`F`, `M`), `dob` |
| `ehr_visits` | `visit_id`, `mrn`, `visit_code` (`WELL`, `ER`, `ADMIT`), `visit_status` (`D` done, `X` cancelled), `start_ts`, `end_ts` |

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW ehr_patients AS
SELECT * FROM VALUES
  ('M001', 'F', DATE'1958-04-12'),
  ('M002', 'M', DATE'1983-09-30'),
  ('M003', 'F', DATE'2001-01-17'),
  ('M004', 'M', DATE'1947-06-05')
  AS t(mrn, sex, dob);

CREATE OR REPLACE TEMP VIEW ehr_visits AS
SELECT * FROM VALUES
  ('V1', 'M001', 'WELL',  'D', TIMESTAMP'2026-02-03 09:00:00', TIMESTAMP'2026-02-03 09:30:00'),
  ('V2', 'M001', 'ER',    'D', TIMESTAMP'2026-06-14 22:10:00', TIMESTAMP'2026-06-15 03:40:00'),
  ('V3', 'M002', 'WELL',  'D', TIMESTAMP'2025-11-20 10:00:00', TIMESTAMP'2025-11-20 10:20:00'),
  ('V4', 'M002', 'ADMIT', 'D', TIMESTAMP'2026-03-08 14:00:00', TIMESTAMP'2026-03-12 11:00:00'),
  ('V5', 'M002', 'ER',    'D', TIMESTAMP'2026-08-01 01:15:00', TIMESTAMP'2026-08-01 05:00:00'),
  ('V6', 'M003', 'WELL',  'X', TIMESTAMP'2026-09-22 08:45:00', TIMESTAMP'2026-09-22 09:05:00')
  AS t(visit_id, mrn, visit_code, visit_status, start_ts, end_ts);

SELECT * FROM ehr_visits ORDER BY visit_id;

## 2. Map it to FHIR

Each view returns the data contract CQL reads: `resource`, a struct in the FHIR R4 shape, and `patient_id`. The
struct holds only the fields the CQL reads; the compiler checks every path the CQL reads against it, so a field
the CQL reads but the struct leaves out fails the compile.

| Source | FHIR | How |
|---|---|---|
| `mrn` | `Patient.id`, `Encounter.subject.reference` | As is; `Patient/<mrn>` for the reference |
| `sex` | `Patient.gender` | `F` to `female`, `M` to `male` |
| `dob` | `Patient.birthDate` | `yyyy-MM-dd` |
| `visit_code` | `Encounter.type.coding` | A crosswalk from local codes to SNOMED CT |
| `visit_status` | `Encounter.status` | `D` to `finished`, `X` to `cancelled` |
| `start_ts`, `end_ts` | `Encounter.period` | FHIR `dateTime` text |

The crosswalk is a table, so a reviewer reads the code mapping as data rather than in a `CASE`.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW visit_type_crosswalk AS
SELECT * FROM VALUES
  ('WELL',  '185349003', 'Encounter for check up (procedure)'),
  ('ER',    '50849002',  'Emergency room admission (procedure)'),
  ('ADMIT', '32485007',  'Hospital admission (procedure)')
  AS t(visit_code, snomed_code, snomed_display);

CREATE OR REPLACE TEMP VIEW fhir_patient AS
SELECT
  mrn AS patient_id,
  CAST(named_struct(
         'id', mrn,
         'gender', CASE sex WHEN 'F' THEN 'female' WHEN 'M' THEN 'male' END,
         'birthDate', date_format(dob, 'yyyy-MM-dd'))
       AS STRUCT<id: STRING, gender: STRING, birthDate: STRING>) AS resource
FROM ehr_patients;

CREATE OR REPLACE TEMP VIEW fhir_encounter AS
SELECT
  v.mrn AS patient_id,
  CAST(named_struct(
         'id', v.visit_id,
         'status', CASE v.visit_status WHEN 'D' THEN 'finished' WHEN 'X' THEN 'cancelled' END,
         'subject', named_struct('reference', concat('Patient/', v.mrn)),
         'type', array(named_struct('coding', array(named_struct(
                   'system', 'http://snomed.info/sct', 'code', c.snomed_code, 'display', c.snomed_display)))),
         'period', named_struct('start', date_format(v.start_ts, "yyyy-MM-dd'T'HH:mm:ss.SSS"),
                                'end',   date_format(v.end_ts,   "yyyy-MM-dd'T'HH:mm:ss.SSS")))
       AS STRUCT<
         id: STRING,
         status: STRING,
         subject: STRUCT<reference: STRING>,
         type: ARRAY<STRUCT<coding: ARRAY<STRUCT<system: STRING, code: STRING, display: STRING>>>>,
         period: STRUCT<start: STRING, `end`: STRING>>) AS resource
FROM ehr_visits v
LEFT JOIN visit_type_crosswalk c ON c.visit_code = v.visit_code;

SELECT patient_id, resource FROM fhir_encounter ORDER BY resource.id;

## 3. Point CQL at the mapped views

`use_resolvers` sets where this notebook's `%%cql` cells and `cql()` calls read their data. Temporary views have
no catalog name, so they are passed as DataFrames: the client registers each on this Spark session for the run.

- It runs on Spark, serverless or a cluster. A SQL warehouse can't read a DataFrame, and refuses before the run.
- Results over DataFrames are never kept or reused.
- Value sets and libraries still come from the default environment; this library includes FHIRHelpers from it.

In [0]:
from cql_databricks.compiler import use_resolvers
from cql_databricks.resolvers import DataFrameDataResolver

use_resolvers(data=DataFrameDataResolver(
    {"Patient": spark.table("fhir_patient"), "Encounter": spark.table("fhir_encounter")},
    name="ehr_mapping",
))
print("[Patient] and [Encounter] now read the mapped views")

## 4. The CQL

| Define | Meaning |
|---|---|
| `Check Up Encounters` | Finished check-ups (SNOMED CT 185349003) in 2026 |
| `Acute Encounters` | Finished emergency room (50849002) or hospital (32485007) admissions in 2026 |
| `Had Check Up` | The patient had a check-up in 2026 |
| `Acute Encounter Count` | How many acute encounters the patient had in 2026 |

The CQL speaks only FHIR and SNOMED CT. It knows nothing of `WELL`, `D` or `mrn`.

In [0]:
%%cql
library VisitMapping version '1.0.0'

using FHIR version '4.0.1'

include FHIRHelpers version '4.0.1' called FHIRHelpers

codesystem "SNOMEDCT": 'http://snomed.info/sct'

code "Check up": '185349003' from "SNOMEDCT" display 'Encounter for check up (procedure)'
code "Emergency room admission": '50849002' from "SNOMEDCT" display 'Emergency room admission (procedure)'
code "Hospital admission": '32485007' from "SNOMEDCT" display 'Hospital admission (procedure)'

parameter "Measurement Period" Interval<DateTime>
  default Interval[@2026-01-01T00:00:00.000, @2026-12-31T23:59:59.999]

context Patient

define "Check Up Encounters":
  [Encounter: "Check up"] E
    where E.status = 'finished'
      and E.period during "Measurement Period"

define "Acute Encounters":
  ([Encounter: "Emergency room admission"] union [Encounter: "Hospital admission"]) E
    where E.status = 'finished'
      and E.period during "Measurement Period"

define "Had Check Up":
  exists "Check Up Encounters"

define "Acute Encounter Count":
  Count("Acute Encounters")

## 5. Run it

Expected, one row per patient:

| Patient | `Had Check Up` | `Acute Encounter Count` | Why |
|---|---|---|---|
| M001 | true | 1 | A check-up in February, an ER visit in June |
| M002 | false | 2 | Its check-up was in 2025; an admission and an ER visit in 2026 |
| M003 | false | 0 | Its check-up was cancelled |
| M004 | false | 0 | No visits |

In [0]:
results = cql(library="VisitMapping").exec("Had Check Up", "Acute Encounter Count")
results["Had Check Up"].display()
results["Acute Encounter Count"].display()

## 6. Check the mapping

The CQL answers are only as right as the mapping. To check it, ask the same questions of the **source** in plain
SQL, written without the mapping: local codes, local statuses, raw timestamps. If the two disagree for any patient,
the mapping is wrong (or the CQL is), and the cell fails, listing the patients.

Try it: delete the `ER` row from `visit_type_crosswalk` and run the notebook again. ER visits then map to an
`Encounter` with no SNOMED code, CQL stops counting them, and the check fails for M001 and M002.

In [0]:
expected = spark.sql("""
  SELECT p.mrn AS patient_id,
         coalesce(bool_or(v.visit_code = 'WELL' AND v.visit_status = 'D'
                          AND v.start_ts >= TIMESTAMP'2026-01-01' AND v.end_ts < TIMESTAMP'2027-01-01'), false)
           AS had_check_up,
         count_if(v.visit_code IN ('ER', 'ADMIT') AND v.visit_status = 'D'
                  AND v.start_ts >= TIMESTAMP'2026-01-01' AND v.end_ts < TIMESTAMP'2027-01-01')
           AS acute_visits
    FROM ehr_patients p
    LEFT JOIN ehr_visits v ON v.mrn = p.mrn
   GROUP BY p.mrn
""").collect()

# Each CQL result has one row per patient: `__subject_id` is the patient, `__val` the value.
cql_check_up = {r["__subject_id"]: r["__val"] for r in results["Had Check Up"].collect()}
cql_acute = {r["__subject_id"]: r["__val"] for r in results["Acute Encounter Count"].collect()}

rows = [(r.patient_id, r.had_check_up, cql_check_up.get(r.patient_id), r.acute_visits, cql_acute.get(r.patient_id))
        for r in sorted(expected, key=lambda r: r.patient_id)]
spark.createDataFrame(rows, "patient_id STRING, sql_had_check_up BOOLEAN, cql_had_check_up BOOLEAN, "
                            "sql_acute_visits LONG, cql_acute_visits LONG").display()

disagree = [r[0] for r in rows if r[1] != r[2] or r[3] != r[4]]
if disagree:
    raise AssertionError(f"CQL and the source disagree for {', '.join(disagree)}: check the mapping")
print(f"The mapping checks out: CQL and the source agree for all {len(rows)} patients")

## Next: use the mapping beyond this notebook

Temporary views last as long as this session. To make the mapping the data every run reads, create the views as
tables or views in a schema of your own, then publish them as an environment's data views with
`publish_data_source()` (see the client README, *Publish data into an environment*). Runs then read them by name,
on a SQL warehouse too, and their results can be kept and reused. Keep the check: run it again whenever the source
or the mapping changes.

In [0]:
import json

dbutils.notebook.exit(json.dumps({"ok": True, "notebook": "08_mapping_your_data"}))